In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.19.6
#   kernelspec:
#     display_name: jaspy
#     language: python
#     name: jaspy
# ---

# NB 06 Final figures

Input: the artifacts `preprocessing` (NB00), `fold_models` (NB04), `comparison_and_mcs` and `tail_grid` (NB05).

Only the figures that appear in the report are drawn here (PNG, folder `Pictures 06_final_figures`):

* `spinupcheck.png`, `marginal_precipitation.png`, `marginal_wind_speed.png`, `marginal_divergence.png`, `meandiurnalcycle.png` from the preprocessed data;
* `mcs_survivors.png`: tau(w) of the Acar and neural models in the MCS at alpha = 0.10, averaged over the folds;
* `chi_grid_model_implied.png`: model-implied co-exceedance chi_m(q) of the MCS survivors against the data.

Nothing is refitted or retrained, and no table is written here except that `fit_all_tables` rescales the table files of all notebooks to the text width (last cell).

## Imports

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from run_support import load_artifact, save_artifact, SMOKE
import gate_config as GC
import figure_recipes as FR
from plotting_preprocessing import plot_marginals, plot_seasonal_diurnal, plot_spinup_threshold_sensitivity
from plotting_core import configure, C_PRECIP, C_WIND, C_DIV_VAR, ACAR_TO_DISPLAY_FAMILY

## Folder

In [ ]:
configure('06_final_figures')
import reporting_tables as RT
RT.configure('06_final_figures')

## Load the artifacts

In [ ]:
data = load_artifact("preprocessing")
precip_all_grid, wind_all_grid, div_all_grid = data['precip_all_grid'], data['wind_all_grid'], data['div_all_grid']
times_pd = pd.to_datetime(data['time_valid'])

cmp_art = load_artifact("comparison_and_mcs")
fm = load_artifact("fold_models")
FOLDS = fm['folds']
main = cmp_art['main_table']

## Spin-up

In [ ]:
plot_spinup_threshold_sensitivity(
    data['spinup_results'], data['spinup_thresholds'], threshold_levels=[5],
    filename='spinupcheck.png')

## Marginal distribution plots

In [ ]:
plot_marginals(precip_all_grid, times_pd, name='Precipitation', color=C_PRECIP, unit='mm/3hr', xmin=0)

In [ ]:
plot_marginals(wind_all_grid,   times_pd, name='Wind Speed',    color=C_WIND,   unit='m/s',     xmin=0)

In [ ]:
plot_marginals(div_all_grid,    times_pd, name='Divergence',    color=C_DIV_VAR, unit='s⁻¹')

## Seasonal variability of diurnal cycle

In [ ]:
plot_seasonal_diurnal(
    data['precip_flat'],
    data['wind_flat'],
    data['div_flat'],
    data['month_flat'],
    data['hour_flat'],
)

# Plot

In [ ]:
# MCS survivors (alpha = 0.10): tau(w) of the Acar and neural survivors, averaged over the folds (mcs_survivors.png).
import pickle, time
from tqdm.auto import tqdm
from conditional_copula import FAMILY_INFO, fold_averaged_curve
from run_support import resolve_max_workers

N_EVAL, CI_BOOT = (10, 8) if SMOKE else (30, 200)      # grid points and bootstrap replicates of the fold-averaged curves (small in a smoke run)

ARR, ACAR, TAU = fm['arrays'], fm['acar'], fm['tau_curves']

# MCS survivors (alpha = 0.10)
surv = main[main['in MCS (alpha=0.10)'].astype(bool)]
disp_to_key = {ACAR_TO_DISPLAY_FAMILY.get(k, k.capitalize()): k for k in FAMILY_INFO}
def to_key(name):
    """Family key of an MCS survivor of the Acar group.

    Parameters:
    name (str): Model name, with or without the prefix 'Acar (continuous): '.

    Returns:
    str: Key of the family in FAMILY_INFO.
    """
    s = name.replace('Acar (continuous): ', '').strip()
    return s if s in FAMILY_INFO else disp_to_key[s]
fams = [to_key(n) for n in surv[surv['group'] == 'Acar'].index]
flows = [n for n in surv[surv['group'] == 'Neural'].index if n.startswith(('C-CopFlow', 'C-CODINE')) and all(n in TAU[f] for f in FOLDS)]
print("Acar survivors:", fams, "| neural survivors:", flows)

# Acar curves averaged over the folds, 95% band from replicate-wise fold means
# Cache key includes fitted data, bandwidths and the numerical implementation.
# A --fresh notebook rerun must not silently reuse a curve from an older fit.
import hashlib
from pathlib import Path
_cache_hash = hashlib.sha256(Path('conditional_copula.py').read_bytes())
_cache_hash.update(repr((N_EVAL, CI_BOOT, tuple(FOLDS), tuple(fams))).encode())
for _fold in FOLDS:
    _a = ARR[_fold]
    for _name in ('u1', 'u2', 'w', 'train'):
        _array = np.ascontiguousarray(np.asarray(_a[_name]))
        _cache_hash.update(repr((_fold, _name, _array.shape, str(_array.dtype))).encode())
        _cache_hash.update(_array.tobytes())
    _cache_hash.update(repr(float(_a['b'])).encode())
    for _fam in fams:
        _cache_hash.update(repr((_fold, _fam, float(ACAR[(_fold, _fam)]['h']))).encode())
CACHE = f'tau_bands_cache_{N_EVAL}_{CI_BOOT}_{_cache_hash.hexdigest()[:16]}.pkl' 
try:
    curves = pickle.load(open(CACHE, 'rb')); curves = {k: v for k, v in curves.items() if k in fams}
except Exception:
    curves = {}
t0 = time.time()
for fam in tqdm([f for f in fams if f not in curves], desc='Acar families'):
    curves[fam] = fold_averaged_curve(ARR, ACAR, FOLDS, fam, n_eval=N_EVAL, ci_boot=CI_BOOT, n_jobs=resolve_max_workers(n_tasks=CI_BOOT))
    pickle.dump(curves, open(CACHE, 'wb'))             # saved after every family, so an interrupted run resumes
    print(f'{fam} done, {time.time() - t0:.0f}s elapsed', flush=True)

# Neural curves averaged over the folds on the common w range (linear interpolation of each fold's curve)
lo_f = max(np.min(TAU[f]['w_grid']) for f in FOLDS); hi_f = min(np.max(TAU[f]['w_grid']) for f in FOLDS)
wg = np.linspace(lo_f, hi_f, 200)
flow_avg = {n: np.mean([np.interp(wg, TAU[f]['w_grid'], TAU[f][n]) for f in FOLDS], axis=0) for n in flows}

# Bands. Networks: a bootstrap band would need every network retrained per replicate, so the band is the range over the training
# seeds of the fold-averaged curve (training variability, not sampling uncertainty). ECBC and Gijbels (below): 95% bootstrap bands
# on the same shared stationary-bootstrap draws as the Acar bands.
import tau_bands as TB
flow_bands = {n: (*TB.seed_range_band(fm['tau_curves_seeds'], TAU, FOLDS, n, wg), 'seed') for n in flows}
N_EVAL_NP = 4 if SMOKE else 15                      # grid points at which the nonparametric replicates are evaluated

# Nonparametric survivors (ECBC, Gijbels): tau(w) was saved with the nonparametric fits (w_grid, tau_grid); the band comes from tau_bands.
# Only models that are in the MCS are drawn, found by their model name in the MCS table.
for name in surv.index[surv['group'] == 'Nonparametric']:
    rows = {f: r for (f, k), r in fm['nonparam'].items() if r['model'] == name and f in FOLDS}
    if set(rows) != set(FOLDS):
        print(f"{name}: MCS survivor without tau(w) curves for folds {sorted(set(FOLDS) - set(rows))}; not drawn")
        continue
    label = 'ECBC (Lu and Ghosh 2024)' if 'ECBC' in name else name
    flow_avg[label] = np.mean(
        [np.interp(wg, np.asarray(rows[f]['w_grid'], float), np.asarray(rows[f]['tau_grid'], float), left=np.nan, right=np.nan)
         for f in FOLDS], axis=0)                      # NaN outside a fold's own grid, so no flat extrapolation
    bt = TB.nonparam_boot_band(ARR, FOLDS, 'ecbc' if 'ECBC' in name else 'gijbels', wg, n_eval=N_EVAL_NP, ci_boot=CI_BOOT,
                               n_jobs=resolve_max_workers(n_tasks=CI_BOOT))
    flow_bands[label] = (bt['lo'], bt['hi'], 'ci')
    if bt['n_failed']:
        print(f"{label}: {bt['n_failed']} bootstrap fits failed")

# Save every curve and band that is drawn (long format, one row per model and grid point).
ROWS = []
for fam, d in curves.items():
    ROWS.append(pd.DataFrame(dict(model=ACAR_TO_DISPLAY_FAMILY.get(fam, fam.capitalize()) + ' (Acar)', w=d['x'], tau=d['tau'],
                                  tau_lo=d['tau_lo'], tau_hi=d['tau_hi'], band='95% bootstrap')))
for name, y in flow_avg.items():
    lo_, hi_, kind_ = flow_bands.get(name, (np.full(len(wg), np.nan), np.full(len(wg), np.nan), 'none'))
    ROWS.append(pd.DataFrame(dict(model=name, w=wg, tau=y, tau_lo=lo_, tau_hi=hi_, band={'ci': '95% bootstrap', 'seed': 'seed range'}.get(kind_, 'none'))))
TAU_BANDS = pd.concat(ROWS, ignore_index=True)
save_artifact("tau_bands", curves=curves, flow_avg=flow_avg, flow_bands=flow_bands, w_grid=wg, bands=TAU_BANDS, smoke=SMOKE)
TAU_BANDS.to_csv(RT.LATEX_DIR + '/tau_bands_survivors.csv', index=False)

FR.mcs_survivors(curves, flow_avg, wg, ACAR_TO_DISPLAY_FAMILY, flow_bands=flow_bands)

In [ ]:
# Notebook 06: model-implied co-exceedance chi_m(q) of ALL models in the pooled MCS (alpha = 0.10), upper tail (top row)
# and lower tail (bottom row). Reads only the saved artifacts; nothing is refitted or retrained.
#   upper: chi_m(q) = P_m(U1 > q, U2 > q) / (1 - q)      lower: chi_m(q) = P_m(U1 < 1 - q, U2 < 1 - q) / (1 - q)
import re
from copula_families import make_bicop, interp_theta_col, safe_make_bicop

tg = load_artifact("tail_grid")
GRID_FULL, MI, SURV = tg['grid_full'], tg['model_implied'], list(tg['survivors_a010'])
PARAM, ACAR, ARR = fm['param'], fm['acar'], fm['arrays']
QS = GC.Q_GRID
REGIMES = (('all', 'all time steps'), ('low', 'low divergence'), ('high', 'high divergence'))

def diag_cdf(cop, a):
    """Copula cdf on the diagonal, C(a, a).

    Parameters:
    cop: Fitted copula with a cdf method.
    a (float): Level on both margins.

    Returns:
    float: The cdf value.
    """
    return float(np.asarray(cop.cdf(np.array([[a, a]])))[0])

def p_cop(cop, q, tail):
    """Joint-exceedance probability of a copula in one tail.

    Upper tail: P(U1 > q, U2 > q) = 1 - 2q + C(q, q). Lower tail: P(U1 < 1 - q, U2 < 1 - q) = C(1 - q, 1 - q).

    Parameters:
    cop: Fitted copula with a cdf method.
    q (float): Threshold of the upper tail; 1 - q is used for the lower tail.
    tail (str): 'upper' or 'lower'.

    Returns:
    float: The probability, clipped to [0, 1].
    """
    c = diag_cdf(cop, q if tail == 'upper' else 1 - q)
    return float(np.clip(1 - 2 * q + c, 0, 1)) if tail == 'upper' else float(np.clip(c, 0, 1))

def row_cop(row):
    """Rebuild a stored parametric fit as a copula object.

    Parameters:
    row (dict): Stored fit with the keys 'key', 'params' and 'rotation'.

    Returns:
    The copula object of the family, parameters and rotation.
    """
    return make_bicop(row['key'], row['params'], rotation=int(row['rotation']))

def step_p(model, f, q, tail):
    """Per-step joint-exceedance probability of a model in one fold.

    Parameters:
    model (str): Model name from the MCS table.
    f (str): Fold name.
    q (float): Threshold of the upper tail; 1 - q is used for the lower tail.
    tail (str): 'upper' or 'lower'.

    Returns:
    np.ndarray: Probability per step, NaN outside the test rows; None if the model cannot be rebuilt from the artifacts.
    """
    A = ARR[f]; te = np.asarray(A['test'], bool); hi = np.asarray(A['high'], bool); n = len(te)
    p = np.full(n, np.nan)
    if model.startswith('Acar (continuous): '):
        fam = model.split(': ', 1)[1]
        r = ACAR[(f, fam)]; curve, fixed = r['curve'], r['fixed']
        w = np.asarray(A['w'], float)
        wg = np.linspace(w[te].min(), w[te].max(), 120)            # theta(w) is smooth: evaluate on a grid, interpolate
        pg = []
        for wv in wg:
            th = float(interp_theta_col(fam, wv, curve['x'].values, curve['theta'].values))
            cop = safe_make_bicop(fam, [th, float(np.clip(fixed[0], 2.001, 49.0))] if fam == 'student' else [th])
            pg.append(p_cop(cop, q, tail))
        p[te] = np.interp(w[te], wg, pg); return p
    m = re.fullmatch(r'Divergence split: (.+) \(low\) / (.+) \(high\)', model)
    if m:
        lo = {r['name']: r for r in PARAM[f]['rows_regime']['low']}[m.group(1)]
        hi_ = {r['name']: r for r in PARAM[f]['rows_regime']['high']}[m.group(2)]
        p[te & ~hi] = p_cop(row_cop(lo), q, tail); p[te & hi] = p_cop(row_cop(hi_), q, tail); return p
    if model.startswith('Unconditional: ') and 'CODINE' not in model:
        row = {r['name']: r for r in PARAM[f]['rows_uncond']}[model.split(': ', 1)[1]]
        p[te] = p_cop(row_cop(row), q, tail); return p
    return None

def chi_curves(model, tail):
    """Model-implied co-exceedance chi_m(q) on the grid QS, by regime.

    The step probabilities are averaged over the test steps and pooled over the folds, as in model_diagnostics.model_implied_chi;
    chi_m(q) is the pooled probability divided by 1 - q (Coles, Heffernan, & Tawn, 1999, Extremes, 2(4), 339-365).

    Parameters:
    model (str): Model name from the MCS table.
    tail (str): 'upper' or 'lower'.

    Returns:
    dict: Regime ('all', 'low', 'high') -> array over QS; None if the model cannot be rebuilt from the artifacts.
    """
    out = {r: [] for r, _ in REGIMES}
    for q in QS:
        num = {r: 0.0 for r, _ in REGIMES}; den = {r: 0 for r, _ in REGIMES}
        for f in FOLDS:
            p = step_p(model, f, q, tail)
            if p is None:
                return None
            te = np.asarray(ARR[f]['test'], bool); hi = np.asarray(ARR[f]['high'], bool)
            for r, sel in (('all', te), ('low', te & ~hi), ('high', te & hi)):
                v = p[sel]; v = v[np.isfinite(v)]; num[r] += v.sum(); den[r] += len(v)
        for r, _ in REGIMES:
            out[r].append(num[r] / den[r] / (1 - q) if den[r] else np.nan)
    return {r: np.array(v) for r, v in out.items()}

CURVES, skipped = {}, []
for m in SURV:
    up, lo = chi_curves(m, 'upper'), chi_curves(m, 'lower')
    if up is None:
        skipped.append(m); continue
    CURVES[m] = dict(upper=up, lower=lo)
    ref = MI[(MI.model == m) & (MI.regime == 'all')].sort_values('q').chi_model.values        # check against notebook 05
    if len(ref) == len(QS):
        print(f"{m}: max |upper(all) - notebook 05| = {np.max(np.abs(up['all'] - ref)):.4f}")
if skipped:
    print("NOT plotted (a network or unsupported model, cannot be rebuilt from the artifacts here):", skipped)

FR.chi_grid_model_implied(GRID_FULL, CURVES, QS)


## Table layout

Every table with a tabular is wrapped so that it shrinks to the text width when it is wider and is never enlarged (`reporting_tables.fit_all_tables`); no table needs a manual size edit in the report.

In [ ]:
import reporting_tables
print('table files fitted to the text width:', reporting_tables.fit_all_tables())